# すべてをまとめて実行

## インストール、インポート

In [ ]:
# !pip install quantecon
# !pip install numba
# !pip install interpolation


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import numpy as np
from quantecon.optimize import brentq
from quantecon.markov import rouwenhorst, tauchen
from interpolation import interp
from numba import njit, guvectorize
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import scipy

## Aiyagariの設定

### Calibration

In [3]:
# パラメータ
beta = 0.96
gamma = 3.0
rho = 0.6
sigma = 0.4
alpha = 0.36
delta = 0.08
b = 3.0

### 政策関数の計算のための関数

In [4]:
# Maliarグリッド

@njit
def maliar_grid(a_min, a_max, N, theta):
		a_grid = np.empty(N)
		for i in range(1,N+1):
				a_grid[i-1] = a_min + (a_max - a_min)*((i-1)/(N-1))**theta

		return a_grid

In [5]:
# exponentialグリッド

@njit
def exponential_grid(a_min, a_max, N):
    """
    対数空間を使ったグリッド生成関数
    Parameters:
        a_min: float, グリッドの最小値
        a_max: float, グリッドの最大値
        N: int, グリッドの分割数
    Returns:
        grid: np.ndarray, 生成されたグリッド
    """
    # 対数変換
    log_a_min = np.log1p(a_min)  # log(1 + a_min)
    log_a_max = np.log1p(a_max)  # log(1 + a_max)

    # 等間隔のグリッドを生成
    log_grid = np.linspace(log_a_min, log_a_max, N)

    # 再変換して結果を格納
    grid = np.expm1(log_grid)  # exp(x) - 1
    return grid

In [6]:
# 補間のための関数（外挿あり）
@guvectorize(['void(float64[:], float64[:], float64[:], float64[:])'], '(n),(nq),(n)->(nq)')
def interpolate_y(x, xq, y, yq):
    """Efficient linear interpolation exploiting monotonicity.
    Extrapolates linearly when xq out of domain of x.
    """
    nxq, nx = xq.shape[0], x.shape[0]

    xi = 0
    x_low = x[0]
    x_high = x[1]
    for xqi_cur in range(nxq):
        xq_cur = xq[xqi_cur]
        while xi < nx - 2:
            if x_high >= xq_cur:
                break
            xi += 1
            x_low = x_high
            x_high = x[xi + 1]

        xqpi_cur = (x_high - xq_cur) / (x_high - x_low)
        yq[xqi_cur] = xqpi_cur * y[xi] + (1 - xqpi_cur) * y[xi + 1]

In [7]:
# 解く問題の設定を行う（パラメータ, グリッド, 効用関数や生産関数）
# モデルを変更する場合にはここを修正
class Setting:

    def __init__(self,
                 R=1.01,                          # 粗実質利子率
                 beta=0.99,                          # 割引因子
                 gamma=1,                          # 相対的リスク回避度(異時点間の代替弾力性の逆数)
                 b=3,                             # 内生的な状態変数の最小値, 借入制約
                 a_max=16,                        # 内生的な状態変数の最大値
                 na=21,                           # 内生的な状態変数のグリッド数
                 na_sd = 800,                     # 定常分布における資本のグリッド数
                 tau = 0.1,                       # 資本課税率
                 Xi = 0.1,                        # 所得移転額
                 mu=0,                            # 外生変数のAR(1)過程の定数項
                 rho=0.6,                        # 外生変数のAR(1)過程の慣性
                 sigma=0.4,                       # 外生変数のAR(1)過程のショック項の標準偏差
                 nz = 11,                         # 外生変数のグリッド数
                 w = 0,                         # 賃金
                 lambdaPF = 1):                 # 政策関数の更新度

        # パラメータを設定する
        self.R = R
        self.beta = beta
        self.b = b
        self.gamma = gamma
        self.na_sd = na_sd
        self.tau = tau
        self.Xi = Xi


        # 外生変数の遷移確率とグリッドを設定する

        #1 rowenhorst
        # mc = rouwenhorst(nz, rho, sigma, mu)
        # self.Pz = mc.P
        # self.z_grid = np.exp(mc.state_values)

        #2 tauchen
        mc = tauchen(nz, rho, sigma, mu, n_std = 2)
        self.Pz = np.array(mc.P)
        if mc.state_values is None:
           raise ValueError("mc.state_values is None, cannot apply np.exp.")
        self.z_grid = np.exp(mc.state_values)


        # 内生的な状態変数のグリッドを設定する
        # a_grid = np.linspace(-b, a_max, na)
        a_grid = maliar_grid(-b, a_max, na, 2)
        #a_grid = exponential_grid(-b, a_max, na)
        self.a_grid = a_grid


        # 賃金を設定
        self.w = w

        # 政策関数の更新度を設定する
        self.lambdaPF = lambdaPF

        # CRRA型効用関数と限界効用を定義する
        gamma = self.gamma
        if gamma == 1:
            self.utility = np.log
            self.mutility = njit(lambda x: 1 / x)
        else:
            self.utility = njit(lambda x: x**(1-gamma) / (1 - gamma))
            self.mutility = njit(lambda x: x**(-gamma))


		    # 政策関数の初期値を定義する
        z_grid = self.z_grid
        hfun_old = np.empty((len(a_grid), len(z_grid)))
        for i_a, a in enumerate(a_grid):
            for i_z, z in enumerate(z_grid):
                c_max = 0.5* (R * a + z + b)
                hfun_old[i_a, i_z] = c_max
        self.hfun_old = hfun_old


# 特定のアルゴリズムを実行して政策関数を更新する関数を出力する
# FOCを変更する場合やアルゴリズムを変更する場合はここを修正

# 今期の状態変数について繰り返し記号はi, 来期の状態変数について繰り返し記号はj

def EGMIterationWithGov(hp): # hpはSettingクラスからつくられるインスタンス

    # インスタンスからローカル変数を定義する
    R, beta, b, gamma, mutility, tau  = hp.R, hp.beta, hp.b, hp.gamma, hp.mutility, hp.tau
    a_grid, z_grid, Pz, w, Xi = hp.a_grid, hp.z_grid, hp.Pz, hp.w, hp.Xi

    R_net = 1 + (1-tau) * (R - 1.00)  # after-tax interest

    @njit
    def UpdatePF(h_old):

        h_new = np.empty_like(h_old)
        a_tilde = np.empty_like(a_grid)
        aprime_new = np.empty_like(a_grid)

        for i_z in range(len(z_grid)): # 今期の生産性について繰り返し
            z = z_grid[i_z]

            for j_a in range(len(a_grid)): # 来期の資産について繰り返し
                aprime = a_grid[j_a]

                expectation = 0
                for j_z in range(len(z_grid)): # 来期の生産性について繰り返し

                    # オイラー方程式の右辺を計算する
                    expectation += mutility(h_old[j_a, j_z]) * Pz[i_z, j_z]

                rhs = R_net * beta * expectation
                c_tilde = rhs**(-1.0/gamma)

                # 今期の資産を計算
                a_tilde[j_a] = (aprime - z*w - Xi + c_tilde)/R_net


            # 来期の資産の政策関数をinterpolationで求める
            interpolate_y(a_tilde, a_grid, a_grid, aprime_new)

            for i_a in range(len(a_grid)): # 今期の資産について繰り返し

                # 来期の資産の政策関数をinterpolationで求める
                #aprime_new = interp(a_tilde, a_grid, a_grid[i_a])

                # 借り入れ制約を考慮しつつ来期の資産を求める
                if aprime_new[i_a] < -b:
                    aprime_new[i_a] = -b
                else:
                    aprime_new[i_a] = aprime_new[i_a]

                # 消費の政策関数を求める
                h_new[i_a,i_z] = R_net * a_grid[i_a] + z*w + Xi - aprime_new[i_a]

        return h_new

    return UpdatePF



# メイン関数：特定のアルゴリズムでiterationを行い、問題を解く関数
# 基本的には変更する必要がない

def SolveProblem(hp,               # Settingクラスからつくられるインスタンス
                Algorithm,         # アルゴリズムを指定
                tol=1e-4,          # 許容繰り返し誤差
                max_iter=10000,     # iteration回数の最大値
                verbose=True,      # 進捗を表示するかどうか
                print_skip=25):    # 進捗を何回ごとに表示するか


    # インスタンスからローカル変数を定義する
    # R, beta, b, mutility = hp.R, hp.beta, hp.b, hp.mutility
    # a_grid, z_grid, Pz = hp.a_grid, hp.z_grid, hp.Pz
    lambdaPF = hp.lambdaPF
    hfun_old = hp.hfun_old

    # チェックのために外生変数のグリッドと遷移確率を表示する
    # print(f"About exogenous variables:")
    # print(f"grid is {z_grid}.")
    # print(f"Transition matrix is {Pz}.")



    # 政策関数を更新する関数を取得する
    UpdatePF = Algorithm(hp)

    # iterationを行い、問題を解く
    i = 0
    error = tol + 1

    while i < max_iter and error > tol:

        # 政策関数を更新する
        hfun_new_tilde = UpdatePF(hfun_old)

        # 古い政策関数と加重平均する
        hfun_new = lambdaPF*hfun_new_tilde + (1-lambdaPF)*hfun_old

        error = np.max(np.abs(hfun_new-hfun_old))
        i += 1
        if verbose and i % print_skip == 0:      # 進捗をprint_skip回ごとに表示する
            print(f"PF Error at iteration {i} is {error}.")
        hfun_old = hfun_new

    if i == max_iter:
        print("Failed to converge!")

    if verbose and i < max_iter:
        print(f"\nConverged in {i} iterations.")

    return hfun_new

### 定常分布の計算のための関数

In [8]:
from numba import njit
import numpy as np


@njit
def gridlookup2(x0, xgrid):
    nx = np.shape(xgrid)[0]
    ix = 0
    for jx in range(nx):
        if x0 <= xgrid[jx]:
            break
        ix += 1
    ix = min(max(1, ix), nx-1)
    return ix - 1

#sd = solve_sd(sd_grid, len(hp.a_grid), len(hp.z_grid),len(a_grid_sd),a_grid, a_grid_sd, hp.Pz, hfun_aprime)


@njit
def solve_sd(mu0: np.ndarray,
                        na: int,
                        nz: int,
                        na_sd: int,    # mu0.shape[0]のこと
                        agrid: np.ndarray,
                        agrid_sd: np.ndarray,
                        Pz: np.ndarray,
                        pfgrid: np.ndarray,
                        critmu: float = 1e-6):

    # -1. 政策関数の行列を補間して、行数を na 個から na_sd 個に変える
    # pfgrid -> pfgrid_new
    pfgrid_new = np.empty((na_sd,nz))              # 空の箱を用意
    for z_index in range(nz):
      aprime_new = np.empty(na_sd)
      interpolate_y(agrid, agrid_sd, pfgrid[:, z_index], aprime_new)
      pfgrid_new[:,z_index] = aprime_new


    # 0. transition matrixを計算
    G = np.zeros((na_sd*nz,na_sd*nz))
    weight = np.zeros((na_sd,nz))

    for iz in range(nz):
      for ia in range(na_sd):

        # 政策関数 pfgrid[ia, iz] に合わせる
        aprime = pfgrid_new[ia, iz]


        if aprime < agrid_sd[0]:
            weight[ia, iz] = 1.0
            jtilde = 0
        elif aprime > agrid_sd[-1]:
            weight[ia, iz] = 0
            jtilde = na_sd - 2
        else:
          # aprimeが落ちる区間の左端のグリッドのインデックスを探す
          jtilde = gridlookup2(aprime, agrid_sd)
          # 重み weight[ia, iz] に格納
          weight[ia, iz] = (agrid_sd[jtilde+1] - aprime) / (agrid_sd[jtilde+1] - agrid_sd[jtilde])

        # スタックしたときの現在の状態のインデックスを計算
        i_s = iz * na_sd + ia

        # 次期外生状態 jz でループ
        for jz in range(nz):

            # スタックしたときの次期の状態のインデックスを計算
            j_s  = jz * na_sd + jtilde

            # 遷移確率行列 G に代入
            #   - Pe[iz, jz] : 現在 iz から次期 jz への外生ショックの遷移確率
            G[i_s, j_s ] += weight[ia, iz]   * Pz[iz, jz]
            G[i_s, j_s + 1] += (1.0 - weight[ia, iz]) * Pz[iz, jz]


    diffmu = 1.0e4
    mu1    = np.zeros((na_sd, nz))

    # 分布の初期値をスタックしてベクトル化
    mu0 = mu0.T.flatten()

    while diffmu > 1e-6:

        # 1. 分布の更新 : dist_new = G' * dist
        mu1 = G.T @ mu0

        # 2. 収束判定
        diffmu = np.max(np.abs(mu1 - mu0))

        # 3. normalizeして次のループへ
        mu0 = mu1 / np.sum(mu1)

    sd = mu0.reshape(nz, na_sd).T
    return sd


### 総労働供給を求める

In [9]:
# 生産性の定常分布を求める

hp = Setting(nz = 7, rho = rho, sigma = sigma) # 生産性の定常分布を求めるために必要ない変数については初期値のまま

muZ_old = np.ones(len(hp.z_grid)) / len(hp.z_grid)

tol=1E-11
maxit=100^4

for it in range(maxit):
  muZ_new = muZ_old @ hp.Pz
  if np.max(np.abs(muZ_new - muZ_old)) < tol: break
  muZ_old = muZ_new

# 総労働供給を求める
Lbar = sum(muZ_new*hp.z_grid)
print(Lbar)

1.125707392526809


## 初期定常状態(ini)実行

In [10]:
import time

start_time = time.time()  # 計測開始

r0 = 0.029
tau = 0.0

diff = 1
loop = 0
while abs(diff) > 1e-6:
    loop += 1
    # print("loop: ", loop)

    # 1
    K0d = ((r0 + delta) / alpha) ** (1 / (alpha - 1)) * Lbar
    w0 = (1 - alpha) * (K0d/Lbar) ** alpha
    Xi0 = tau * r0 * K0d

    # 2
    hp = Setting(beta=beta, gamma = gamma, b = b, a_max = 45, lambdaPF = 1, na = 50, na_sd= 50, nz = 7, rho = rho, sigma = sigma, R = 1 + r0, w = w0, Xi = Xi0, tau=tau)
    hfun_c = SolveProblem(hp,EGMIterationWithGov)

    # 3
    # hfun_c から 次期のアセット aの政策関数を求める
    hfun_aprime = np.empty((len(hp.a_grid), len(hp.z_grid)))
    a_mesh, z_mesh = np.meshgrid(hp.a_grid, hp.z_grid, indexing='ij') # ユニバーサル関数を使用するためのグリッドを生成
    R_net = 1.0 + (1.0 - tau) * r0
    hfun_aprime = R_net * a_mesh + hp.w * z_mesh - hfun_c


    # 定常分布用のグリッドを用意
    a_grid_sd = np.linspace(-hp.b, hp.a_grid[-1], hp.na_sd)
    # a_grid_sd = maliar_grid(-hp.b , hp.a_grid[-1], hp.na_sd, 2)

    # 初期の定常分布を定義
    sd_grid = np.full((len(a_grid_sd), len(hp.z_grid)),
                      1 / (len(a_grid_sd) * len(hp.z_grid))
                      )

    sd = solve_sd(sd_grid, len(hp.a_grid), len(hp.z_grid),len(a_grid_sd),hp.a_grid, a_grid_sd, hp.Pz, hfun_aprime)

    # 4
    #Amesh, _ = np.meshgrid(a_grid_sd, hp.z_grid, indexing='ij')
    #K0s = np.sum(Amesh * sd)
    # --- hfun_aprime を補間 ---
    hfun_aprime_interp = np.empty((len(a_grid_sd), len(hp.z_grid)))
    for iz in range(len(hp.z_grid)):
        interpolate_y(hp.a_grid, a_grid_sd, hfun_aprime[:, iz], hfun_aprime_interp[:, iz])
    # 総資本ストック供給 K_s の計算
    K0s = np.sum(hfun_aprime_interp * sd)

    # print("K0s: ", K0s)
    diff = (K0s - K0d)
    # print("diff: ", diff)
    # print("r0: ", r0)

    r0 = r0 - 0.001 * diff

r0 = r0 + 0.001 * diff


end_time = time.time()  # 計測終了

print("Elapsed time:", end_time - start_time, "seconds")

PF Error at iteration 25 is 0.06564909355203952.
PF Error at iteration 50 is 0.013325259987844618.
PF Error at iteration 75 is 0.004002687388442894.
PF Error at iteration 100 is 0.0013100284285911812.
PF Error at iteration 125 is 0.00042415677028628807.
PF Error at iteration 150 is 0.00013129958279023413.

Converged in 156 iterations.
PF Error at iteration 25 is 0.06567180297006558.
PF Error at iteration 50 is 0.013344167705511722.
PF Error at iteration 75 is 0.00402211533071295.
PF Error at iteration 100 is 0.0013270939572151974.
PF Error at iteration 125 is 0.00043582211939963145.
PF Error at iteration 150 is 0.00013761030211867364.

Converged in 157 iterations.
PF Error at iteration 25 is 0.06567795104987795.
PF Error at iteration 50 is 0.013349146188609495.
PF Error at iteration 75 is 0.004027264718054369.
PF Error at iteration 100 is 0.0013316773442184626.
PF Error at iteration 125 is 0.00043901082541708547.
PF Error at iteration 150 is 0.00013937048050394196.

Converged in 158 it

In [11]:
sd_ini =  sd
k_ini = K0d

w = (1 - alpha) * (K0d/Lbar) ** alpha
w_ini = w
r_ini = r0

xi_ini = tau * r0 * K0d

aprime_ini = np.squeeze(hfun_aprime)

c_ini = np.squeeze(hfun_c)

## 最終定常状態(fin)実行

In [12]:
import time

start_time = time.time()  # 計測開始

r0 = 0.029
# tau = 0.1
tau = 0.01
diff = 1
loop = 0
while abs(diff) > 1e-6:
    loop += 1
    # print("loop: ", loop)

    # 1
    K0d = ((r0 + delta) / alpha) ** (1 / (alpha - 1)) * Lbar
    w0 = (1 - alpha) * (K0d/Lbar) ** alpha
    Xi0 = tau * r0 * K0d

    # 2
    hp = Setting(beta=beta, gamma = gamma, b = b, a_max = 45, lambdaPF = 1, na = 50, na_sd= 50, nz = 7, rho = rho, sigma = sigma, R = 1 + r0, w = w0, Xi = Xi0, tau=tau)
    hfun_c = SolveProblem(hp,EGMIterationWithGov)

    # 3
    # hfun_c から 次期のアセット aの政策関数を求める
    hfun_aprime = np.empty((len(hp.a_grid), len(hp.z_grid)))
    a_mesh, z_mesh = np.meshgrid(hp.a_grid, hp.z_grid, indexing='ij') # ユニバーサル関数を使用するためのグリッドを生成
    R_net = 1.0 + (1.0 - tau) * r0
    hfun_aprime = R_net * a_mesh + hp.w * z_mesh - hfun_c


    # 定常分布用のグリッドを用意
    a_grid_sd = np.linspace(-hp.b, hp.a_grid[-1], hp.na_sd)
    # a_grid_sd = maliar_grid(-hp.b , hp.a_grid[-1], hp.na_sd, 2)

    # 初期の定常分布を定義
    sd_grid = np.full((len(a_grid_sd), len(hp.z_grid)),
                      1 / (len(a_grid_sd) * len(hp.z_grid))
                      )

    sd = solve_sd(sd_grid, len(hp.a_grid), len(hp.z_grid),len(a_grid_sd),hp.a_grid, a_grid_sd, hp.Pz, hfun_aprime)

    # 4
    #Amesh, _ = np.meshgrid(a_grid_sd, hp.z_grid, indexing='ij')
    #K0s = np.sum(Amesh * sd)
    # --- hfun_aprime を補間 ---
    hfun_aprime_interp = np.empty((len(a_grid_sd), len(hp.z_grid)))
    for iz in range(len(hp.z_grid)):
        interpolate_y(hp.a_grid, a_grid_sd, hfun_aprime[:, iz], hfun_aprime_interp[:, iz])
    # 総資本ストック供給 K_s の計算
    K0s = np.sum(hfun_aprime_interp * sd)

    # print("K0s: ", K0s)
    diff = (K0s - K0d)
    # print("diff: ", diff)
    # print("r0: ", r0)

    r0 = r0 - 0.001 * diff

r0 = r0 + 0.001 * diff


end_time = time.time()  # 計測終了

print("Elapsed time:", end_time - start_time, "seconds")

PF Error at iteration 25 is 0.06562970984242611.
PF Error at iteration 50 is 0.013312638826320722.
PF Error at iteration 75 is 0.003990641884868751.
PF Error at iteration 100 is 0.0012996942477414564.
PF Error at iteration 125 is 0.0004172191425766414.
PF Error at iteration 150 is 0.00012762166252855422.

Converged in 156 iterations.
PF Error at iteration 25 is 0.06566582345102745.
PF Error at iteration 50 is 0.013343058048469914.
PF Error at iteration 75 is 0.00402181206438712.
PF Error at iteration 100 is 0.001326923401471447.
PF Error at iteration 125 is 0.00043568835129548233.
PF Error at iteration 150 is 0.00013752942014377822.

Converged in 157 iterations.
PF Error at iteration 25 is 0.06567637061222342.
PF Error at iteration 50 is 0.013351570195382578.
PF Error at iteration 75 is 0.004030621432661974.
PF Error at iteration 100 is 0.0013347789801159138.
PF Error at iteration 125 is 0.0004411530624963689.
PF Error at iteration 150 is 0.00014055782263255878.

Converged in 158 itera

In [13]:
sd_fin =  sd
k_fin = K0d

w = (1 - alpha) * (K0d/Lbar) ** alpha
w_fin = w
r_fin = r0
xi_fin = tau * r0 * K0d

aprime_fin = np.squeeze(hfun_aprime)

c_fin = np.squeeze(hfun_c)

## 移行過程

まずは三期間で考える

### 最終定常状態(fin)の価値関数を導出

In [14]:
T = 30 # iniからfinまでの期間を設定
tau = np.zeros(T) # T期間分のtauを格納する箱を作る
tau[1:] = 0.01 # 初期状態以外のtauを0.1に設定

K_list = np.linspace(k_ini, k_fin, T) # 資本ストックの初期値を当て推量、最初と最後の値は定常状態のものを用いる
w = (1 - alpha) * (K_list/Lbar) ** alpha # 当て推量したKを基に賃金を計算
r = alpha * (K_list / Lbar) ** (alpha - 1) - delta # 当て推量したKを基に金利を計算
xi = K_list * tau * r # 当て推量したK,rとtauより政府移転を計算

In [15]:
value_function_list = np.empty((T,len(hp.a_grid), len(hp.z_grid))) # 計算した価値関数を格納する箱を設定
v_old_box = np.zeros((len(hp.a_grid), len(hp.z_grid))) # 繰り返し計算前の価値関数を格納する箱を設定
v_tilde_box = np.empty((len(hp.a_grid), len(hp.z_grid))) # 繰り返し計算後の価値関数を格納する箱を設定

diff = 1
loop = 0
while diff > 1e-6: # 繰り返し計算前後で価値関数が等しくなるまで
    loop += 1
    for i_a in range(len(hp.a_grid)): # 今期の資産グリッドで繰り返し
        for i_z in range(len(hp.z_grid)): # 今期の生産性で繰り返し
            aprime = (1 + (1 - tau[-1]) * r_fin) * hp.a_grid[i_a] + w_fin * hp.z_grid[i_z] + tau[-1] * r_fin * k_fin - c_fin[i_a, i_z] #
            value = 0
            for j_z in range(len(hp.z_grid)): # 次期の生産性で繰り返し
                v_tilde_interp = np.interp(aprime, hp.a_grid, v_old_box[:, j_z])
                # interpolate_y(hp.a_grid, aprime, v_old_box[:, j_z], v_tilde_box)
                value += v_tilde_interp * hp.Pz[i_z, j_z]
            v_tilde_box[i_a, i_z] = hp.utility(c_fin[i_a, i_z]) + beta * value

    diff = np.max(np.abs(v_tilde_box - v_old_box))
    # print("diff: ", diff)
    v_old_box = v_tilde_box # 価値関数を更新
    v_tilde_box = np.empty((len(hp.a_grid), len(hp.z_grid))) # 次のループの計算のために空にする
    v_tilde_box_interp = np.empty((len(hp.a_grid), len(hp.z_grid))) # 次のループの計算のために空にする

value_function_list[-1] = v_old_box # 計算した価値関数を格納、これがfinでの価値関数となる



### 収束するまで繰り返し

### 4-a政策関数と価値関数の導出

In [16]:
# 何を引数として何を出力してくれる関数なのかをそれぞれに記載する。
from scipy.optimize import fminbound
from numba import jit
import numpy as np

# maximization_problem関数の定義（JIT化）
# @jit(nopython=True)
def maximization_problem(c, i_a,  i_z, vfun_next, t):
    """ 今期の消費、資産、状態、次期の価値関数を用いて、今期の最大化問題を定義する関数
    Args:
    c : 今期の消費
    i_a : 今期の資産
    i_z : 今期の状態
    vfun_next : 次期の価値関数
    t : 今期の期数
    """

    aprime = (1 + (1 - tau[t-1]) * r[t-1]) * hp.a_grid + w[t-1] * hp.z_grid[i_z] + tau[t-1] * r[t-1] * K_list[t-1] - c
    expectation = 0.0

    for j_z in range(len(hp.z_grid)):  # 次期の生産性
        value_function_interp = np.empty(len(hp.a_grid), dtype=np.float64)
        interpolate_y(hp.a_grid, aprime, vfun_next[:, j_z], value_function_interp) #interpolate_yはベクトルを返すもの
        expectation += value_function_interp[i_a] * hp.Pz[i_z, j_z]

    u_value = (hp.utility(c) + beta * expectation)  # brent_maxをつかうなら最大化するために負にしない
    return u_value

# 消費cを最大化する関数を定義
def maximize_u_value(vfun_next):
    optimal_c_list = np.zeros((len(hp.a_grid), len(hp.z_grid)))  # 最適な消費を格納するための箱（初期化）
    v_value_list = np.zeros((len(hp.a_grid), len(hp.z_grid)))  # 価値関数の箱（初期化）

    for i_a in range(len(hp.a_grid)):

        for i_z in range(len(hp.z_grid)):

            # objective関数を定義（rhsを最大化するために負にする）
            def objective(c):
                return -maximization_problem(c, i_a, i_z, vfun_next, t)

            # fminboundを使ってcを最適化
            c_min = 1e-8
            c_max = (1 + (1 - tau[t-1]) * r[t-1]) * hp.a_grid[i_a] + w[t-1] * hp.z_grid[i_z] + b + tau[t-1] * r[t-1] * K_list[t-1] #消費の最大値（借入制約に引っかかったとき）
            sol = fminbound(objective, c_min, c_max, xtol=1e-05, full_output = True)
            # print(sol)

            optimal_c = sol[0]
            v_value = sol[1]

            # 最適な消費と価値関数の値を格納
            optimal_c_list[i_a, i_z] = optimal_c # 最適な消費を代入
            v_value_list[i_a, i_z] = -v_value #価値関数の値を代入


    return optimal_c_list, v_value_list  # 両方を返す

### 4-b

In [17]:
@njit
def split_prob_to_agrid(aprime: float, agrid: np.ndarray) -> np.ndarray:
    """ 政策関数によって得られた a の値が、a_grid のどの点に移動するのかを示す確率ベクトルを計算する関数

    Args:
    aprime: 政策関数によって得られた a の値
    agrid: a のグリッド
    return: aprime が agrid のどの点に移動するのかを示す確率ベクトル
    """
    probs = np.zeros(len(agrid))
    # a の値が a_grid の最小値より小さい場合
    if aprime <= agrid[0]:
        probs[0] = 1.0
        return probs

    # a の値が a_grid の最大値より大きい場合
    if aprime >= agrid[-1]:
        probs[-1] = 1.0
        return probs

    # a の値が a_grid の最小値より大きく最大値より小さい場合
    for i in range(len(agrid)-1):
        if agrid[i] <= aprime <= agrid[i+1]:
            probs[i] = (agrid[i+1] - aprime) / (agrid[i+1] - agrid[i])
            probs[i+1] = (aprime - agrid[i]) / (agrid[i+1] - agrid[i])
            return probs

    # この行までに return されていない場合はエラーを出す
    raise ValueError("Splitting probability failed.")

@njit
def calc_weight_matrix(aprime_grid: np.ndarray, agrid: np.ndarray, split_prob_to_agrid = split_prob_to_agrid) -> np.ndarray:
    """ 前期の定常分布に係る重みづけ確率行列 weight_matrix を計算する関数

    Args:
        aprime_grid: 政策関数の行き先のグリッド
        agrid: 資産のグリッド
        split_prob: 政策関数によって得られた a の値が、a_grid のどの点に移動するのかを示す確率ベクトルを計算する関数

    Returns:
        weight_matrix: 前期の定常分布に係る重みづけ確率のグリッド、shape は aprime_grid と同じ
    """
    weight_matrix = np.empty((len(aprime_grid), len(aprime_grid[0]), len(agrid)))

    # aprime_grid は 二次元配列
    for i in range(len(aprime_grid)):
        for j in range(len(aprime_grid[0])):
            weight_matrix[i, j] = split_prob_to_agrid(float(aprime_grid[i, j]), agrid)
    return weight_matrix

@njit
def gen_pmesh(aprime_grid: np.ndarray, P: np.ndarray, next_y_index: int) -> np.ndarray:
    """ 遷移確率を 政策関数 aprime_grid との要素ごとの計算に使えるように meshgrid を生成する関数

    Args:
        aprime_grid (np.ndarray): 政策関数のグリッド
        P (np.ndarray): 遷移確率行列
        next_y_index (int): 次期の所得状態 y' のインデックス

    Returns:
        np.ndarray: 政策関数と遷移確率行列を使って計算するための meshgrid
    """
    pmesh = np.zeros(aprime_grid.shape)
    for i in range(len(aprime_grid)):
        for j in range(len(aprime_grid[0])):
            pmesh[i, j] = P[j, next_y_index]
    return pmesh

# def calc_sd_point(Pmesh, weight, sd_old) -> float:
#     """ 定常分布の一点を計算する関数

#     Args:
#         Pmesh (np.ndarray): 政策関数と遷移確率行列を使って計算した meshgrid
#         weight (np.ndarray): 前期の定常分布に係る重みづけ確率
#         sd_old (np.ndarray): 一期前の定常分布 f(a, y)

#     Returns:
#         float: 更新された(次期の)定常分布の一点の値
#     """
#     return np.sum(Pmesh * weight * sd_old)

@njit
def update_sd(aprime_grid: np.ndarray, agrid: np.ndarray, sd_old: np.ndarray, P: np.ndarray,
            split_prob_to_agrid = split_prob_to_agrid, calc_weight_matrix = calc_weight_matrix,
            gen_P = gen_pmesh,
            # calc_sd_point = calc_sd_point
            ) -> np.ndarray:
    """ 定常分布を更新する関数

    Args:
        aprime_grid (np.ndarray): 政策関数のグリッド
        agrid (np.ndarray): 資産のグリッド
        sd_old (np.ndarray): 更新前の定常分布
        P (np.ndarray): 遷移確率行列
        split_prob (Callable): 政策関数によって得られた a の値が、a_grid のどの点に移動するのかを示す確率ベクトルを計算する関数
        calc_weight (Callable): 前期の定常分布に係る重みづけ確率 weight を計算する関数
        gen_P (Callable): 遷移確率を 政策関数 aprime_grid との要素ごとの計算に使えるように meshgrid を生成する関数
        calc_sd_point (Callable): 定常分布の一点を計算する関数

    Returns:
        sd_new（np.ndarray）: 更新された定常分布のグリッド
    """
    sd_new = np.zeros(sd_old.shape)
    weight_matrix = calc_weight_matrix(aprime_grid, agrid, split_prob_to_agrid)
    for i in range(len(aprime_grid)):
        for j in range(len(aprime_grid[0])):
            p_mesh = gen_P(aprime_grid, P, j)
            # sd_new[i, j] = calc_sd_point(p_mesh, weight_matrix[:,:,i], sd_old)
            sd_new[i,j] = np.sum(p_mesh * weight_matrix[:,:,i] * sd_old)

    return sd_new

@njit
def sd_calculation(sd0: np.ndarray, aprime_grid: np.ndarray, agrid: np.ndarray, P: np.ndarray,

                        update_sd = update_sd) -> np.ndarray:
    """ 定常分布を計算する関数

    Args:
        aprime_grid (np.ndarray): 政策関数のグリッド
        agrid (np.ndarray): 資産のグリッド
        guess_sd (np.ndarray): 初期の定常分布のグリッド
        P (np.ndarray): 遷移確率行列
        tol (float, optional): 許容する誤差. Defaults to 1e-6.
        max_iter (int, optional): 最大の反復回数. Defaults to 1000.
        update_sd (_type_, optional): 定常分布を更新する関数. Defaults to update_sd.

    Returns:
        np.ndarray: 収束した定常分布のグリッド
    """
    sd_old = sd0
    sd_next = update_sd(aprime_grid, agrid, sd_old, P)
    return sd_next

In [18]:
from numba import njit
import numpy as np

@njit
def sd_calculation(sd_old: np.ndarray,
                   aprime_grid: np.ndarray,
                   agrid: np.ndarray,
                   Pz: np.ndarray) -> np.ndarray:
    """
    solve_sd() のロジックを用いて、1回だけ次期の定常分布を計算する関数。

    Parameters:
    - sd_old: np.ndarray, 現在の分布 (na, nz)
    - aprime_grid: np.ndarray, 政策関数 (na, nz)
    - agrid: np.ndarray, 資産グリッド (na,)
    - Pz: np.ndarray, 所得ショックの遷移行列 (nz, nz)

    Returns:
    - sd_next: np.ndarray, 次期の分布 (na, nz)
    """
    na, nz = aprime_grid.shape
    G = np.zeros((na * nz, na * nz))
    weight = np.zeros((na, nz))

    for iz in range(nz):
        for ia in range(na):
            aprime = aprime_grid[ia, iz]

            if aprime < agrid[0]:
                weight[ia, iz] = 1.0
                jtilde = 0
            elif aprime > agrid[-1]:
                weight[ia, iz] = 0.0
                jtilde = na - 2
            else:
                for i in range(len(agrid) - 1):
                    if agrid[i] <= aprime <= agrid[i + 1]:
                        jtilde = i
                        weight[ia, iz] = (agrid[i + 1] - aprime) / (agrid[i + 1] - agrid[i])
                        break

            i_s = iz * na + ia
            for jz in range(nz):
                j_s = jz * na + jtilde
                G[i_s, j_s] += weight[ia, iz] * Pz[iz, jz]
                G[i_s, j_s + 1] += (1.0 - weight[ia, iz]) * Pz[iz, jz]

    # sd_old をベクトル化して更新
    mu_old = sd_old.T.flatten()  # shape = (na * nz,)
    mu_new = G.T @ mu_old
    mu_new = mu_new / np.sum(mu_new)  # 正規化

    # 2次元 (na, nz) に戻して返す
    sd_next = mu_new.reshape(nz, na).T
    return sd_next

### 4-c

In [19]:
def compute_K_path(sd_list: list, a_grid: np.ndarray) -> list:
    """
    分布リスト sd_list をもとに、各期の総資本 K_t を計算する関数

    Parameters
    ----------
    sd_list : list of ndarray
        各期の分布（shape: (na, nz)）のリスト
    a_grid : ndarray
        資産グリッド（shape: (na,)）

    Returns
    -------
    K_path : list of float
        各期の総資本 K_t のリスト
    """
    K_path = []

    for mu in sd_list:
        K_t = np.sum(mu * a_grid[:, np.newaxis])  # 資産×分布の要素積
        K_path.append(float(K_t))

    return K_path

### 4-d

In [20]:
def compute_K_error(K_path_new: list, K_path_old: list) -> float:
    """
    新しく求めた K_path と当初の予測 K_path_old との最大誤差を返す関数

    Parameters
    ----------
    K_path_new : list of float
        新しく計算された各期の総資本（K_path）
    K_path_old : list of float
        当て推量された各期の総資本

    Returns
    -------
    max_error : float
        各期の誤差の絶対値の最大値
    """
    K_path_new = np.array(K_path_new)
    K_path_old = np.array(K_path_old)

    error = (K_path_new - K_path_old)
    max_abs_error = np.max(np.abs(error))

    return max_abs_error, error

### 実行

In [21]:
T = 30

tau = np.zeros(T) # T期間分のtauを格納する箱を作る
tau[1:] = 0.01 # 初期状態以外のtauを0.1に設定

K_list = np.linspace(k_ini, k_fin, T) # 資本ストックの初期値を当て推量、最初と最後の値は定常状態のものを用いる


# print(K_list)
max_abs_error = 1
loop = 0

while max_abs_error > 1e-6: # 繰り返し計算前後で価値関数が等しくなるまで
      loop = loop + 1
# for loop in range(10):

      w = (1 - alpha) * (K_list/Lbar) ** alpha # 当て推量したKを基に賃金を計算
      r = alpha * (K_list / Lbar) ** (alpha - 1) - delta # 当て推量したKを基に金利を計算
      # print(f"r:{r}")
      xi = K_list * tau * r # 当て推量したK,rとtauより政府移転を計算

      hfun_aprime = np.empty((T,len(hp.a_grid), len(hp.z_grid)))
      hfun_c = np.empty((T,len(hp.a_grid), len(hp.z_grid)))

      for t in range(T-1,-1,-1):
          # t期の政策関数を求める
          hfun_c[t-1], value_function_list[t-1] = maximize_u_value(value_function_list[t])

          for i_a in range(len(hp.a_grid)):
              for i_z in range(len(hp.z_grid)):
                hfun_aprime[t-1, i_a, i_z] = (hp.a_grid[i_a] * (1 + (1 - tau[t-1]) * r[t-1]) + w[t-1] * hp.z_grid[i_z] + tau[t-1] * r[t-1] * K_list[t-1] - hfun_c[t-1, i_a, i_z]) # ここまでが行き

      hfun_aprime[0] = aprime_ini

      sd_list = np.empty((T, len(hp.a_grid), len(hp.z_grid))) # sd_listはT＋1期分、初期状態が第0期とするため、ここからが戻り
      sd_list[0] = sd_ini.copy() # sd_listの一番最初はsd_ini、それ以外はのちの計算で埋まるためemptyのまま

      for t in range(1,T):
          sd_next = sd_calculation(sd_list[t-1], hfun_aprime[t-1], hp.a_grid, hp.Pz) # 1期前の分布より、今期の分布を求めている
          sd_list[t] = sd_next # 今期の分布をsd_listに保存

      K_pass = compute_K_path(sd_list, hfun_aprime)

      max_abs_error , error = compute_K_error(K_pass, K_list)

      K_list = K_list + error * 0.05
      K_list[0] = k_ini
      # K_list[-1] = k_fin　# 教科書では、1回目以降のイタレーションでK_finを使わない

      # print(f" k_pass: {k_pass}")

      print(f" ループ回数: {loop}")
      print(f" 最大誤差 (max_abs_error): {max_abs_error:.6e}")



 ループ回数: 1
 最大誤差 (max_abs_error): 1.042477e+02
 ループ回数: 2
 最大誤差 (max_abs_error): 9.496871e+01
 ループ回数: 3
 最大誤差 (max_abs_error): 5.998369e+01
 ループ回数: 4
 最大誤差 (max_abs_error): 4.565270e+01
 ループ回数: 5
 最大誤差 (max_abs_error): 4.054376e+01
 ループ回数: 6
 最大誤差 (max_abs_error): 3.753514e+01
 ループ回数: 7
 最大誤差 (max_abs_error): 3.501586e+01
 ループ回数: 8
 最大誤差 (max_abs_error): 3.261007e+01
 ループ回数: 9
 最大誤差 (max_abs_error): 3.025646e+01
 ループ回数: 10
 最大誤差 (max_abs_error): 2.790195e+01
 ループ回数: 11
 最大誤差 (max_abs_error): 2.556459e+01
 ループ回数: 12
 最大誤差 (max_abs_error): 2.327029e+01
 ループ回数: 13
 最大誤差 (max_abs_error): 2.102513e+01
 ループ回数: 14
 最大誤差 (max_abs_error): 1.882039e+01
 ループ回数: 15
 最大誤差 (max_abs_error): 1.671831e+01
 ループ回数: 16
 最大誤差 (max_abs_error): 1.473728e+01
 ループ回数: 17
 最大誤差 (max_abs_error): 1.287796e+01
 ループ回数: 18
 最大誤差 (max_abs_error): 1.115522e+01
 ループ回数: 19
 最大誤差 (max_abs_error): 9.561113e+00
 ループ回数: 20
 最大誤差 (max_abs_error): 8.106145e+00
 ループ回数: 21
 最大誤差 (max_abs_error): 6.794002e+00
 ループ回数: 22
 最大誤差 (max_

KeyboardInterrupt: 

# テスト

In [ ]:
K_list

In [ ]:


T = 3
hfun_aprime = np.empty((T,len(hp.a_grid), len(hp.z_grid)))
hfun_c = np.empty((T,len(hp.a_grid), len(hp.z_grid)))
tmp = np.empty((T,len(hp.a_grid), len(hp.z_grid)))


for t in range(T-1,-1,-1):
    # t期の政策関数を求める
    hfun_c[t-1], value_function_list[t-1] = maximize_u_value(value_function_list[t])

    for i_a in range(len(hp.a_grid)):
        for i_z in range(len(hp.z_grid)):
          hfun_aprime[t-1, i_a, i_z] = (hp.a_grid[i_a] * (1 + (1 - tau[t-1]) * r[t-1]) + w[t-1] * hp.z_grid[i_z] + tau[t-1] * r[t-1] * K_list[t-1] - hfun_c[t-1, i_a, i_z])

In [ ]:
hfun_c[1][:,0]

In [ ]:
hfun_aprime[1][:,0]

In [ ]:
  # 政策関数をプロット
df = pd.DataFrame({
    'x_axis': hp.a_grid,
    'y_axis': hfun_aprime[2][:,0]
})

# plot
plt.plot('x_axis', 'y_axis', data=df, linestyle='-', marker='o')
plt.show()

In [ ]:
  # 政策関数をプロット
df = pd.DataFrame({
    'x_axis': hp.a_grid,
    'y_axis': hfun_c[2][:,0]
})

# plot
plt.plot('x_axis', 'y_axis', data=df, linestyle='-', marker='o')
plt.show()

In [ ]:
I = 3
for i in range(1,I):
  print(i)

# 4-b分布の計算(過去)

In [ ]:
@njit
def split_prob_to_agrid(aprime: float, agrid: np.ndarray) -> np.ndarray:
    """ 政策関数によって得られた a の値が、a_grid のどの点に移動するのかを示す確率ベクトルを計算する関数

    Args:
    aprime: 政策関数によって得られた a の値
    agrid: a のグリッド
    return: aprime が agrid のどの点に移動するのかを示す確率ベクトル
    """
    probs = np.zeros(len(agrid))
    # a の値が a_grid の最小値より小さい場合
    if aprime <= agrid[0]:
        probs[0] = 1.0
        return probs

    # a の値が a_grid の最大値より大きい場合
    if aprime >= agrid[-1]:
        probs[-1] = 1.0
        return probs

    # a の値が a_grid の最小値より大きく最大値より小さい場合
    for i in range(len(agrid)-1):
        if agrid[i] <= aprime <= agrid[i+1]:
            probs[i] = (agrid[i+1] - aprime) / (agrid[i+1] - agrid[i])
            probs[i+1] = (aprime - agrid[i]) / (agrid[i+1] - agrid[i])
            return probs

    # この行までに return されていない場合はエラーを出す
    raise ValueError("Splitting probability failed.")

@njit
def calc_weight_matrix(aprime_grid: np.ndarray, agrid: np.ndarray, split_prob_to_agrid = split_prob_to_agrid) -> np.ndarray:
    """ 前期の定常分布に係る重みづけ確率行列 weight_matrix を計算する関数

    Args:
        aprime_grid: 政策関数の行き先のグリッド
        agrid: 資産のグリッド
        split_prob: 政策関数によって得られた a の値が、a_grid のどの点に移動するのかを示す確率ベクトルを計算する関数

    Returns:
        weight_matrix: 前期の定常分布に係る重みづけ確率のグリッド、shape は aprime_grid と同じ
    """
    weight_matrix = np.empty((len(aprime_grid), len(aprime_grid[0]), len(agrid)))

    # aprime_grid は 二次元配列
    for i in range(len(aprime_grid)):
        for j in range(len(aprime_grid[0])):
            weight_matrix[i, j] = split_prob_to_agrid(float(aprime_grid[i, j]), agrid)
    return weight_matrix

@njit
def gen_pmesh(aprime_grid: np.ndarray, P: np.ndarray, next_y_index: int) -> np.ndarray:
    """ 遷移確率を 政策関数 aprime_grid との要素ごとの計算に使えるように meshgrid を生成する関数

    Args:
        aprime_grid (np.ndarray): 政策関数のグリッド
        P (np.ndarray): 遷移確率行列
        next_y_index (int): 次期の所得状態 y' のインデックス

    Returns:
        np.ndarray: 政策関数と遷移確率行列を使って計算するための meshgrid
    """
    pmesh = np.zeros(aprime_grid.shape)
    for i in range(len(aprime_grid)):
        for j in range(len(aprime_grid[0])):
            pmesh[i, j] = P[j, next_y_index]
    return pmesh

# def calc_sd_point(Pmesh, weight, sd_old) -> float:
#     """ 定常分布の一点を計算する関数

#     Args:
#         Pmesh (np.ndarray): 政策関数と遷移確率行列を使って計算した meshgrid
#         weight (np.ndarray): 前期の定常分布に係る重みづけ確率
#         sd_old (np.ndarray): 一期前の定常分布 f(a, y)

#     Returns:
#         float: 更新された(次期の)定常分布の一点の値
#     """
#     return np.sum(Pmesh * weight * sd_old)

@njit
def update_sd(aprime_grid: np.ndarray, agrid: np.ndarray, sd_old: np.ndarray, P: np.ndarray,
            split_prob_to_agrid = split_prob_to_agrid, calc_weight_matrix = calc_weight_matrix,
            gen_P = gen_pmesh,
            # calc_sd_point = calc_sd_point
            ) -> np.ndarray:
    """ 定常分布を更新する関数

    Args:
        aprime_grid (np.ndarray): 政策関数のグリッド
        agrid (np.ndarray): 資産のグリッド
        sd_old (np.ndarray): 更新前の定常分布
        P (np.ndarray): 遷移確率行列
        split_prob (Callable): 政策関数によって得られた a の値が、a_grid のどの点に移動するのかを示す確率ベクトルを計算する関数
        calc_weight (Callable): 前期の定常分布に係る重みづけ確率 weight を計算する関数
        gen_P (Callable): 遷移確率を 政策関数 aprime_grid との要素ごとの計算に使えるように meshgrid を生成する関数
        calc_sd_point (Callable): 定常分布の一点を計算する関数

    Returns:
        sd_new（np.ndarray）: 更新された定常分布のグリッド
    """
    sd_new = np.zeros(sd_old.shape)
    weight_matrix = calc_weight_matrix(aprime_grid, agrid, split_prob_to_agrid)
    for i in range(len(aprime_grid)):
        for j in range(len(aprime_grid[0])):
            p_mesh = gen_P(aprime_grid, P, j)
            # sd_new[i, j] = calc_sd_point(p_mesh, weight_matrix[:,:,i], sd_old)
            sd_new[i,j] = np.sum(p_mesh * weight_matrix[:,:,i] * sd_old)

    return sd_new

@njit
def sd_calculation(sd0: np.ndarray, aprime_grid: np.ndarray, agrid: np.ndarray, P: np.ndarray,

                        update_sd = update_sd) -> np.ndarray:
    """ 定常分布を計算する関数

    Args:
        aprime_grid (np.ndarray): 政策関数のグリッド
        agrid (np.ndarray): 資産のグリッド
        guess_sd (np.ndarray): 初期の定常分布のグリッド
        P (np.ndarray): 遷移確率行列
        tol (float, optional): 許容する誤差. Defaults to 1e-6.
        max_iter (int, optional): 最大の反復回数. Defaults to 1000.
        update_sd (_type_, optional): 定常分布を更新する関数. Defaults to update_sd.

    Returns:
        np.ndarray: 収束した定常分布のグリッド
    """
    sd_old = sd0
    sd_next = update_sd(aprime_grid, agrid, sd_old, P)
    return sd_next

# 正しい解があるかどうかのテスト

In [ ]:
K_list

array([7.20629651, 6.95864747, 6.71099843])

In [ ]:
T = 3

tau = np.zeros(T) # T期間分のtauを格納する箱を作る
tau[1:] = 0.1 # 初期状態以外のtauを0.1に設定

# K_list = np.linspace(k_ini, k_fin, T) # 資本ストックの初期値を当て推量、最初と最後の値は定常状態のものを用いる

K_test = np.linspace(k_ini, k_fin, 50) # 資本ストックの初期値を当て推量、最初と最後の値は定常状態のものを用いる



print(K_list)
max_abs_error = 1
loop = 0

# while max_abs_error > 1e-6: # 繰り返し計算前後で価値関数が等しくなるまで
for loop in range(len(K_test)):

      K_list = [k_ini,K_test[loop], k_fin]

      w = (1 - alpha) * (K_list/Lbar) ** alpha # 当て推量したKを基に賃金を計算
      r = alpha * (K_list / Lbar) ** (alpha - 1) - delta # 当て推量したKを基に金利を計算
      # print(f"r:{r}")
      xi = K_list * tau * r # 当て推量したK,rとtauより政府移転を計算

      hfun_aprime = np.empty((T,len(hp.a_grid), len(hp.z_grid)))
      hfun_c = np.empty((T,len(hp.a_grid), len(hp.z_grid)))

      for t in range(T-1,-1,-1):
          # t期の政策関数を求める
          hfun_c[t-1], value_function_list[t-1] = maximize_u_value(value_function_list[t])

          for i_a in range(len(hp.a_grid)):
              for i_z in range(len(hp.z_grid)):
                hfun_aprime[t-1, i_a, i_z] = (hp.a_grid[i_a] * (1 + (1 - tau[t-1]) * r[t-1]) + w[t-1] * hp.z_grid[i_z] + tau[t-1] * r[t-1] * K_list[t-1] - hfun_c[t-1, i_a, i_z])

      hfun_aprime[0] = aprime_ini

      sd_list = np.empty((T, len(hp.a_grid), len(hp.z_grid))) # sd_listはT＋1期分、初期状態が第0期とするため
      sd_list[0] = sd_ini.copy() # sd_listの一番最初はsd_ini、それ以外はのちの計算で埋まるためemptyのまま

      for t in range(1,T):
          sd_next = sd_calculation(sd_list[t-1], hfun_aprime[t-1], hp.a_grid, hp.Pz) # 1期前の分布より、今期の分布を求めている
          sd_list[t] = sd_next # 今期の分布をsd_listに保存

      sd_list[-1] = sd_fin

      k_pass = compute_K_path(sd_list, hfun_aprime)

      max_abs_error , error = compute_K_error(k_pass, K_list)

      print(f" error: {error}")

      print(f" ループ回数: {loop}")



[np.float64(7.206296512460907), np.float64(6.710998428190012), np.float64(6.710998428190012)]
 error: [-6.27168105 -5.6638483  -6.35482674]
 ループ回数: 0
 error: [-6.27881684 -5.66087342 -6.36190762]
 ループ回数: 1
 error: [-6.28403321 -5.65606375 -6.36701688]
 ループ回数: 2
 error: [-6.28805199 -5.65007566 -6.37092107]
 ループ回数: 3
 error: [-6.29118919 -5.64319492 -6.37395399]
 ループ回数: 4
 error: [-6.29373333 -5.6357071  -6.37640642]
 ループ回数: 5
 error: [-6.29584338 -5.62777016 -6.37843825]
 ループ回数: 6
 error: [-6.29761896 -5.61948409 -6.38014873]
 ループ回数: 7
 error: [-6.29912533 -5.6109148  -6.38160202]
 ループ回数: 8
 error: [-6.30041663 -5.60211883 -6.38285085]
 ループ回数: 9
 error: [-6.30153393 -5.59313945 -6.38393492]
 ループ回数: 10
 error: [-6.30250713 -5.58400814 -6.38488293]
 ループ回数: 11
 error: [-6.30335928 -5.57474915 -6.3857169 ]
 ループ回数: 12
 error: [-6.30410855 -5.56538162 -6.3864541 ]
 ループ回数: 13
 error: [-6.30476693 -5.55591703 -6.38710593]
 ループ回数: 14
 error: [-6.30534496 -5.54637089 -6.38768067]
 ループ回数: 15
 err